In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# 1. Define Constants (Standard Hodgkin-Huxley Parameters)
C_m  =   1.0   # Membrane Capacitance (uF/cm^2)
g_K  =  36.0   # Maximum Potassium Conductance (mS/cm^2)
g_Na = 120.0   # Maximum Sodium Conductance (mS/cm^2)
g_L  =   0.3   # Maximum Leak Conductance (mS/cm^2)
E_K  = -77.0   # Potassium Reversal Potential (mV)
E_Na =  50.0   # Sodium Reversal Potential (mV)
E_L  = -54.387 # Leak Reversal Potential (mV)

# 2. Define Gating Functions (Alpha and Beta)
def alpha_m(V): return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
def beta_m(V):  return 4.0 * np.exp(-(V + 65.0) / 18.0)
def alpha_h(V): return 0.07 * np.exp(-(V + 65.0) / 20.0)
def beta_h(V):  return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
def alpha_n(V): return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
def beta_n(V):  return 0.125 * np.exp(-(V + 65.0) / 80.0)

# 3. External Injection Current (The Stimulus)
def I_inj(t):
    # Inject a 10 uA/cm^2 current pulse between t=5 and t=6 ms
    if 5.0 < t < 6.0:
        return 10.0
    return 0.0

# 4. The Hodgkin-Huxley Differential Equations
def dALLdt(X, t):
    V, m, h, n = X

    # Calculate ionic currents
    I_Na = g_Na * m**3 * h * (V - E_Na)
    I_K  = g_K  * n**4 * (V - E_K)
    I_L  = g_L  * (V - E_L)

    # Membrane potential derivative
    dVdt = (I_inj(t) - I_Na - I_K - I_L) / C_m

    # Gating variables derivatives
    dmdt = alpha_m(V) * (1.0 - m) - beta_m(V) * m
    dhdt = alpha_h(V) * (1.0 - h) - beta_h(V) * h
    dndt = alpha_n(V) * (1.0 - n) - beta_n(V) * n

    return dVdt, dmdt, dhdt, dndt

# 5. Setup Time and Initial Conditions
t = np.arange(0.0, 20.0, 0.01) # Time vector (0 to 20 ms)
V0 = -65.0 # Initial resting potential
m0 = alpha_m(V0) / (alpha_m(V0) + beta_m(V0))
h0 = alpha_h(V0) / (alpha_h(V0) + beta_h(V0))
n0 = alpha_n(V0) / (alpha_n(V0) + beta_n(V0))
X0 = [V0, m0, h0, n0]

# 6. Run the Simulation (Solve ODEs)
X = odeint(dALLdt, X0, t)
V = X[:, 0]

# 7. Plotting the Results
plt.figure(figsize=(10, 6))

# Plot Membrane Potential
plt.plot(t, V, 'k', linewidth=2, label='Membrane Potential')

# Add Annotations for the Report
plt.annotate('Stimulus Applied', xy=(5, -60), xytext=(2, -30),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))
plt.annotate('Depolarization\n(Na+ Influx)', xy=(6, -10), xytext=(1.5, 0))
plt.annotate('Peak', xy=(6.8, 35), xytext=(4, 30),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))
plt.annotate('Repolarization\n(K+ Efflux)', xy=(8, -20), xytext=(9, 10))
plt.annotate('Hyperpolarization', xy=(12, -75), xytext=(12, -50),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

# Formatting
plt.title('Simulated Action Potential of the Hodgkin-Huxley Model', fontsize=14, fontweight='bold')
plt.xlabel('Time (ms)', fontsize=12)
plt.ylabel('Membrane Potential (mV)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.axhline(y=-65, color='r', linestyle=':', label='Resting Potential')
plt.legend()
plt.tight_layout()

# Save the figure as a high-res image for your document
plt.savefig('action_potential.png', dpi=300)
plt.show()